# EuroSAT-Sentinel: Phase 1 Exploratory Data Analysis, Training & Explainability Walkthrough
**University Deep Learning & Computer Vision Project — Phase 1**

This notebook provides an interactive walkthrough of the **EuroSAT-Sentinel** Phase 1 pipeline:
1. Configuration & Reproducible Seed Setup
2. Stratified Dataset Loading & Exploratory Data Analysis (EDA)
3. Classical Machine Learning Baseline (RGB Histograms + Spatial Gradients $\rightarrow$ Random Forest)
4. Deep Learning Transfer Learning Model (`ResNet-18`) Inspection & Evaluation
5. Grad-CAM Spatial Attention Explainability Visualization

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_config, set_seed, get_device
from src.models.classifier import build_model, count_parameters
from src.inference.predictor import EuroSATPredictor

config = load_config(PROJECT_ROOT / 'configs' / 'default_config.yaml')
set_seed(config['project']['seed'])
device = get_device()
print(f"Project: {config['project']['name']} | Seed: {config['project']['seed']} | Device: {device}")

## 1. Exploratory Data Analysis & Stratified Splits Summary

In [ ]:
eda_path = PROJECT_ROOT / 'outputs' / 'metrics' / 'eda_summary.json'
if eda_path.exists():
    with open(eda_path) as f:
        eda_data = json.load(f)
    print(json.dumps(eda_data, indent=2))

for fig_name in ['eda_class_distribution.png', 'eda_sample_grid.png', 'eda_augmentation_showcase.png']:
    fp = PROJECT_ROOT / 'outputs' / 'figures' / fig_name
    if fp.exists():
        display(Image.open(fp))

## 2. Deep Learning Model Architecture & Classical ML vs. CNN Comparison

In [ ]:
model = build_model(config)
total_p, train_p = count_parameters(model)
print(f"Architecture: {model.architecture} | Total Parameters: {total_p:,} | Trainable: {train_p:,}")

eval_path = PROJECT_ROOT / 'outputs' / 'metrics' / 'test_evaluation_metrics.json'
if eval_path.exists():
    with open(eval_path) as f:
        eval_metrics = json.load(f)
    print(f"Test Top-1 Accuracy: {eval_metrics['test_accuracy_top1']*100:.2f}%")
    print(f"Test Top-3 Accuracy: {eval_metrics['test_accuracy_top3']*100:.2f}%")
    print(f"Test Macro F1-Score: {eval_metrics['test_f1_macro']:.4f}")
    display(pd.DataFrame(eval_metrics['per_class_metrics']).T)

## 3. Training Curves, Confusion Matrix & Grad-CAM Explainability

In [ ]:
for fig_name in ['training_curves.png', 'confusion_matrix.png', 'gradcam_explainability_gallery.png']:
    fp = PROJECT_ROOT / 'outputs' / 'figures' / fig_name
    if fp.exists():
        display(Image.open(fp))